In [3]:
# installing libraries for evaluation
# rouge-score: Google's official ROUGE-implementation
# bert-score: calculates BERTScore via pre-trained BERT-models
! pip install rouge-score bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.9 MB/s eta 0:00:00


In [4]:
# loading all relevant libraries
import pandas as pd                          # for csv
from rouge_score import rouge_scorer         # for ROUGE calculation
from bert_score import score as bert_score   # for BERTScore calculation
import numpy as np 

In [9]:
# loading the ground truth dataset from the course
gt = pd.read_csv("tax-dataset-answers/tax_dataset_answers.csv")
print(gt.head(3))

             id                                             answer
0  CORP-TAX-001  Das Einkommen, das der unbeschränkt Steuerpfli...
1  CORP-TAX-002  Die nicht verrechneten Zinsen stellen eine ver...
2  CORP-TAX-003  Steuerpflichtige, die aufgrund ihrer Rechtsfor...


In [10]:
# loading the results (outputs) from the 3 models
df_inference = pd.read_csv("model-outputs/windisch_inference_output.csv")   # inference model using GPT-2
df_sft       = pd.read_csv("model-outputs/windisch_sft_output.csv")         # fine tuned model using Llama 3.1
df_rag       = pd.read_csv("model-outputs/windisch_rag_output.csv")         # RAG model using Qwen

In [11]:
# checking structure
print("inference columns:", df_inference.columns.tolist())
print("sft columns:", df_sft.columns.tolist())
print("rag columns:", df_rag.columns.tolist())

inference columns: ['id', 'answer']
sft columns: ['id', 'answer']
rag columns: ['id', 'answer']


In [12]:
# merging all 3 model answers with ground truth into one df
# merging by ID column
# renaming the answer column for each model

# renaming answer columns for each model output df
inf_clean = df_inference[["id", "answer"]].rename(columns={"answer": "inf_answer"})
sft_clean = df_sft[["id", "answer"]].rename(columns={"answer": "sft_answer"})
rag_clean = df_rag[["id", "answer"]].rename(columns={"answer": "rag_answer"})

# mergin into one df with ground truth
merged = gt[["id", "answer"]].merge(inf_clean, on="id") \
                              .merge(sft_clean, on="id") \
                              .merge(rag_clean, on="id")

print(merged.head(3))

             id                                             answer  \
0  CORP-TAX-001  Das Einkommen, das der unbeschränkt Steuerpfli...   
1  CORP-TAX-002  Die nicht verrechneten Zinsen stellen eine ver...   
2  CORP-TAX-003  Steuerpflichtige, die aufgrund ihrer Rechtsfor...   

                                          inf_answer  \
0  [In the German version, the question is: "What...   
1  Vielleicht, daß der Zeit des Kompetenz ist zu ...   
2  Zum Verfügung zur Geschichte des Bundes, die Ü...   

                                          sft_answer  \
0  Die Bemessungsgrundlage für die Körperschaftst...   
1  Zinslose Darlehen von einer Kapitalgesellschaf...   
2  Gemäß § 7 Abs. 3 Z 2 EStG sind folgende Körper...   

                                          rag_answer  
0  Die steuerliche Bemessungsgrundlage für die Kö...  
1  Wenn eine Körperschaft ein zinsloses Darlehen ...  
2  Basierend auf den bereitgestellten Gesetzestex...  


In [13]:
import shutil, os

# checking available storage for downloading BERT model
statvfs = os.statvfs("/kaggle/working")
free_gb = (statvfs.f_frsize * statvfs.f_bavail) / (1024**3)
print(f"free storage: {free_gb:.1f} GB")

free storage: 19.5 GB


In [14]:
# computing the rouge score
def compute_rouge(predictions, references):
    
    # Initialize the ROUGE scorer with three variants:
    # rouge1: unigram overlap (single words)
    # rouge2: bigram overlap (pairs of words)
    # rougeL: longest common subsequence (captures sentence-level structure)
    # use_stemmer=True --> words will be reduced to their root form
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True
    )

    # stores F1 scores for each sample
    r1_scores = []
    r2_scores = []
    rL_scores = []

    # iterating over each prediction plus each reference (pairwise)
    for pred, ref in zip(predictions, references):
        # handling missing values
        # converting values to string, else to empty string
        pred = str(pred) if pd.notna(pred) else ""
        ref  = str(ref)  if pd.notna(ref)  else ""

        # computing rouge score 
        # order: score(reference, prediction)
        # reference is ground truth
        # prediction is model output
        result = scorer.score(ref, pred)

        # extracting only the F1 score (=fmeasure)
        # F1 = harmonic mean of precision and recall
        r1_scores.append(result["rouge1"].fmeasure)
        r2_scores.append(result["rouge2"].fmeasure)
        rL_scores.append(result["rougeL"].fmeasure)

    # aggregated results:
    
    return {
        # calculating mean F1 of all samples for each model
        "ROUGE-1 F1":       round(np.mean(r1_scores), 4),
        "ROUGE-2 F1":       round(np.mean(r2_scores), 4),
        "ROUGE-L F1":       round(np.mean(rL_scores), 4),
        # storing for each sample for error analysis
        "_r1_per_sample":   r1_scores,
        "_r2_per_sample":   r2_scores,
        "_rL_per_sample":   rL_scores,
    }

In [21]:
# computing the bertscore 
def compute_bertscore(predictions, references):

    # handling missing values, converting to strings, empty string for NA
    predictions = [str(p) if pd.notna(p) else "" for p in predictions]
    references  = [str(r) if pd.notna(r) else "" for r in references]

    # replacing empty strings with placeholders to avoid bertscore crashing
    predictions = [p if p.strip() != "" else "[EMPTY]" for p in predictions]
    references  = [r if r.strip() != "" else "[EMPTY]" for r in references]

    # computing bertscore
    # function returns precision, recall and F1
    P, R, F1 = bert_score(
        predictions,
        references,
        model_type="distilbert-base-multilingual-cased", # model type = multilingual DistilBERT model
        lang="de",                                       # suitable for german text
        verbose=True, # shows progress bar
        use_fast_tokenizer=True 
    )

    # converts F1 scores to python list
    # each entry = one prediction-reference pair
    f1_list = F1.tolist()

    # returning aggregated results
    return {
        # mean F1 score across all samples for each model
        # represents overall semantic similarity performance
        "BERTScore F1":     round(np.mean(f1_list), 4),
        # storing F1 for each sample for error analysis
        "_bert_per_sample": f1_list,
    }

In [22]:
# defining ground truth column from merged df
gt_answer = "answer"

print("Calculating ROUGE and BERTScore for all 3 models:\n")


# Model 1: GPT-2 inference

# merged["inf_answer"] is the models prediciton
# merged["gt_answer"] is the ground truth
print("modell 1: gpt-2 inference...")
rouge_inf = compute_rouge(merged["inf_answer"], merged[gt_answer]) # computing ROUGE for Model 1
bert_inf  = compute_bertscore(merged["inf_answer"], merged[gt_answer]) # computing BERTScore for Model 1
# printing only aggregated ROUGE results (excluding per-sample values)
# keys starting with "_" contain detailed data (not needed for summary output)
print("  rouge:", {k: v for k, v in rouge_inf.items() if not k.startswith("_")})
# same for BERTScore, only printing overall F1 score
print("  bert:",  {k: v for k, v in bert_inf.items()  if not k.startswith("_")})

# Model 2: Llama fine-tuned

print("\nmodell 2: llama 3.1 fine-tuned...")
rouge_sft = compute_rouge(merged["sft_answer"], merged[gt_answer]) # computing ROUGE for Model 2
bert_sft  = compute_bertscore(merged["sft_answer"], merged[gt_answer]) # computing BERTScore for Model 2
print("  rouge:", {k: v for k, v in rouge_sft.items() if not k.startswith("_")})
print("  bert:",  {k: v for k, v in bert_sft.items()  if not k.startswith("_")})

# Model 3: Qwen RAG

print("\nmodell 3: qwen rag...")
rouge_rag = compute_rouge(merged["rag_answer"], merged[gt_answer]) # computing ROUGE for Model 3
bert_rag  = compute_bertscore(merged["rag_answer"], merged[gt_answer]) # computing BERTScore for Model 3
print("  rouge:", {k: v for k, v in rouge_rag.items() if not k.startswith("_")})
print("  bert:",  {k: v for k, v in bert_rag.items()  if not k.startswith("_")})

Calculating ROUGE and BERTScore for all 3 models:

modell 1: gpt-2 inference...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/19 [00:00<?, ?it/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/19 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/11 [00:00<?, ?it/s]

done in 69.89 seconds, 9.20 sentences/sec
  rouge: {'ROUGE-1 F1': np.float64(0.2352), 'ROUGE-2 F1': np.float64(0.0755), 'ROUGE-L F1': np.float64(0.1593)}
  bert: {'BERTScore F1': np.float64(0.8129)}


In [24]:
# resultstable
# printing table with results with all models and metrics

results_table = pd.DataFrame([
    {
        "Model":        "GPT-2 large (Inference only)",
        "ROUGE-1 F1":   rouge_inf["ROUGE-1 F1"],
        "ROUGE-2 F1":   rouge_inf["ROUGE-2 F1"],
        "ROUGE-L F1":   rouge_inf["ROUGE-L F1"],
        "BERTScore F1": bert_inf["BERTScore F1"],
    },
    {
        "Model":        "Llama 3.1 8B (Fine-Tuned, QLoRA)",
        "ROUGE-1 F1":   rouge_sft["ROUGE-1 F1"],
        "ROUGE-2 F1":   rouge_sft["ROUGE-2 F1"],
        "ROUGE-L F1":   rouge_sft["ROUGE-L F1"],
        "BERTScore F1": bert_sft["BERTScore F1"],
    },
    {
        "Model":        "Qwen2.5 + multilingual-E5 (RAG)",
        "ROUGE-1 F1":   rouge_rag["ROUGE-1 F1"],
        "ROUGE-2 F1":   rouge_rag["ROUGE-2 F1"],
        "ROUGE-L F1":   rouge_rag["ROUGE-L F1"],
        "BERTScore F1": bert_rag["BERTScore F1"],
    },
])

print("\n=== Results Table ===")
print(results_table.to_string(index=False))

# saving as csv
results_table.to_csv("/kaggle/working/windisch_evaluation_results.csv", index=False)
print("\nsaving complete")


=== Results Table ===
                           Model  ROUGE-1 F1  ROUGE-2 F1  ROUGE-L F1  BERTScore F1
    GPT-2 large (Inference only)      0.0821      0.0108      0.0600        0.7743
Llama 3.1 8B (Fine-Tuned, QLoRA)      0.2427      0.0736      0.1654        0.8256
 Qwen2.5 + multilingual-E5 (RAG)      0.2352      0.0755      0.1593        0.8129

saving complete


In [27]:
# putting all per-sample scores into a df (one row per question)
error_df = merged[["id", "answer", "inf_answer", "sft_answer", "rag_answer"]].copy()

# adding the per-sample scores from the calculated dictionaries to the new df
error_df["inf_rouge1"] = rouge_inf["_r1_per_sample"]
error_df["inf_bert"] = bert_inf["_bert_per_sample"]

error_df["sft_rouge1"] = rouge_sft["_r1_per_sample"]
error_df["sft_bert"] = bert_sft["_bert_per_sample"]

error_df["rag_rouge1"] = rouge_rag["_r1_per_sample"]
error_df["rag_bert"] = bert_rag["_bert_per_sample"]

# saving as csv for error analysis
error_df.to_csv("/kaggle/working/windisch_per_sample_scores.csv", index=False)
print(f"saved: {len(error_df)} rows, {len(error_df.columns)} columns")
print(error_df.columns.tolist())

saved: 643 rows, 11 columns
['id', 'answer', 'inf_answer', 'sft_answer', 'rag_answer', 'inf_rouge1', 'inf_bert', 'sft_rouge1', 'sft_bert', 'rag_rouge1', 'rag_bert']


In [30]:
def show_worst(model_col, bert_col, n=5):
    # shows column name for rouge from bert
    rouge_col = bert_col.replace("bert", "rouge1")
    
    # filtering the n worst rows based on bertscore
    worst = error_df.nsmallest(n, bert_col)[
        ["id", "answer", model_col, bert_col, rouge_col]
    ].copy()
    
    # printing results in notebook
    print(f"\nworst {n} examples: {model_col}")
    for _, row in worst.iterrows():
        print(f"\nID: {row['id']}")
        print(f"  GT:     {str(row['answer'])[:200]}")
        print(f"  Model: {str(row[model_col])[:200]}")
        print(f"  BERTScore: {row[bert_col]:.3f} | ROUGE-1: {row[rouge_col]:.3f}")
    
    return worst

# calculating worst examples for each model
worst_inf = show_worst("inf_answer", "inf_bert")
worst_sft = show_worst("sft_answer", "sft_bert")
worst_rag = show_worst("rag_answer", "rag_bert")

# adding a "model" column to the df
worst_inf["model"] = "GPT-2 Inference"
worst_sft["model"] = "Llama 3.1 Fine-Tuned"
worst_rag["model"] = "Qwen RAG"

# merging all examples into one df and saving as csv
worst_all = pd.concat([worst_inf, worst_sft, worst_rag], ignore_index=True)
worst_all.to_csv("/kaggle/working/windisch_worst_examples.csv", index=False)
print(f"\nsaved ({len(worst_all)} rows)")


worst 5 examples: inf_answer

ID: ESTG-NEW-019
  GT:     Der Reverse Stock Split an sich ist für die glatten Anteile kein steuerpflichtiger Vorgang, da sich nur die Stückelung ändert und die Beteiligungsquote unverändert bleibt. Die Anschaffungskosten der a
  Model: If you can't get the answer to the above questions, then please contact us.  If you are a German citizen and you have not received any email in which you can be contacted by us, then you are not eligi
  BERTScore: 0.697 | ROUGE-1: 0.000

ID: ESTG-NEW-012
  GT:     Nein. Kryptowährungen, die zum Betriebsvermögen gehören, fallen nicht unter § 27b EStG, sondern sind nach den allgemeinen Gewinnermittlungsvorschriften der jeweiligen Einkunftsart zu behandeln. Gemäß 
  Model: Gavin Andresen   Offline   Activity: 1652  Merit: 1014   Chief Scientist   LegendaryActivity: 1652Merit: 1014Chief Scientist Re: [ANN] BitcoinDark (BTCD)--a secure, private, untraceable cryptocurrency
  BERTScore: 0.698 | ROUGE-1: 0.000

ID: ESTG27-011
  GT